<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_pixbox_l8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq install aria2 > /dev/null 2>&1
!pip install --quiet omnicloudmask==1.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.8 MB/s eta 0:00:00


In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from cloudband.acquisition import zenodo
from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.eval.report import as_percentages, to_frame
from cloudband.labels import pixbox_l8
from cloudband.pipelines import pixbox_l8 as pipeline

reload_package("cloudband")

In [4]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/data/pixbox_l8")
SCENES_DIR = Path("/content/data/scenes_l8")
PRED_DIR = Path("/content/data/predictions_l8")
RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)

Mounted at /content/drive
project: /content/cloudband
device: cuda:Tesla T4
free disk: 65.4 GiB
omnicloudmask: 1.7.0
rasterio: 1.5.1
pandas: 2.2.3
numpy: 2.1.3


In [5]:
zenodo.download(zenodo.L8_LABELS, DATA_DIR)
!unzip -o -q {DATA_DIR}/{zenodo.L8_LABELS_ARCHIVE} -d {DATA_DIR}

table = pipeline.load_reference(DATA_DIR / zenodo.L8_LABELS_CSV)
print("pixels:", len(table))
print("counts:", pixbox_l8.label_counts(pixbox_l8.label_masks(table)))

pixels: 18830
counts: {'clear': 12365, 'cloud': 5478, 'shadow': 1396}


In [11]:
archive = zenodo.download(zenodo.L8_SCENES, Path("/content/data"))
!unzip -o -q {archive} -d {SCENES_DIR}
!ls {SCENES_DIR} | head
print("entries:", len(list(SCENES_DIR.iterdir())))

LC81960302014022LGN00
LC81970182015080LGN00
LC81970222013186LGN00
LC81970222014109LGN00
LC81980222014260LGN00
LC81980232014276LGN00
LC81990242014075LGN00
LC81990242014107LGN00
LC82030242014103LGN00
LC82030242015058LGN00
entries: 11


In [12]:
expected = sorted(pixbox_l8.PRODUCT_ID_TO_SCENE.values())
found = {p.name for p in SCENES_DIR.rglob("*") if p.is_dir()}
missing = [name for name in expected if not any(name in f for f in found)]
print("expected:", len(expected), "| missing:", missing)
assert not missing, missing

expected: 11 | missing: []


In [13]:
scene = sorted(SCENES_DIR.rglob(f"{expected[0]}*"))[0]
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)

masks = ocm.predict_scenes([scene], PRED_DIR, config, sensor=ocm.LANDSAT8)
pred = pipeline.read_prediction(masks[0])

pid = next(k for k, v in pixbox_l8.PRODUCT_ID_TO_SCENE.items() if v == expected[0])
sub = table[table[pixbox_l8.PRODUCT_COLUMN] == pid]
ref = pixbox_l8.label_masks(sub)
got = pixbox_l8.sample_predictions(sub, pred)

print("shape:", pred.shape)
print("reference:", {k: int(v.sum()) for k, v in ref.items()})
print("predicted:", {c: int((got == c).sum()) for c in range(4)})

Running inference using cuda float32:   0%|          | 0/1 [00:00<?, ?it/s]

shape: (7811, 7681)
reference: {'clear': 629, 'cloud': 1031, 'shadow': 109}
predicted: {0: 768, 1: 712, 2: 219, 3: 51}


In [14]:
scene_paths = [sorted(SCENES_DIR.rglob(f"{name}*"))[0] for name in expected]
masks = ocm.predict_scenes(scene_paths, PRED_DIR, config, sensor=ocm.LANDSAT8)
print("masks:", len(masks))

Running inference using cuda float32:   0%|          | 0/11 [00:00<?, ?it/s]

masks: 11


In [15]:
scored = pipeline.attach_predictions(table, PRED_DIR)
confusions = pipeline.score(scored)
as_percentages(to_frame(confusions))

,tp,tn,fp,fn,ua,pa,oa,boa,f1,iou
experiment,,,,,,,,,,
clear,12322,6031,434,43,96.60,99.65,97.47,96.47,98.10,96.27
cloud,5088,13329,23,390,99.55,92.88,97.81,96.35,96.10,92.49
shadow,941,17412,22,455,97.72,67.41,97.47,83.64,79.78,66.36
